# L14 capstone: a GPT with its own tokenizer

Choose **Runtime, Change runtime type, T4 GPU**, then run the cells in order. You will train a
byte-level BPE tokenizer on Shakespeare, encode the corpus, train a small GPT on the tokens, measure
its validation loss in bits per character (comparable across tokenizers), and sample from it with
temperature, top-k and top-p. The last cell is the autograder.

In [ ]:
import hashlib
import math
import os
import pathlib
import re
import time
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F

SMALL = os.environ.get("SMALL") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

CORPUS_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
CORPUS_SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
# Inside a checkout of the Forge repository the file is already there; in Colab it is downloaded.
local = [p for p in (pathlib.Path("data/tinyshakespeare.txt"), pathlib.Path("../data/tinyshakespeare.txt")) if p.exists()]
raw = local[0].read_bytes() if local else urllib.request.urlopen(CORPUS_URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == CORPUS_SHA256, "unexpected corpus contents"
text = raw.decode("utf-8")
if SMALL:
    text = text[:60_000]
split_at = int(0.9 * len(text))
train_text, val_text = text[:split_at], text[split_at:]
print(f"{len(text):,} characters; device {DEVICE}")

## 1. A byte-level BPE tokenizer

Split the text into chunks with the pattern below (words keep their leading space), count each
distinct chunk once with its frequency, and learn merges of the most frequent adjacent pair within
chunks, starting from the 256 byte values. Encode by applying merges in the order they were learned.

In [ ]:
PATTERN = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+")
VOCAB_SIZE = 300 if SMALL else 512


def merge(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


def train_bpe(text, vocab_size):
    """Return the merges {(a, b): new_id} in the order learned."""
    raise NotImplementedError("Write your code here")


class Tokenizer:
    def __init__(self, merges):
        self.merges = merges
        self.vocab = {i: bytes([i]) for i in range(256)}
        for (a, b), i in merges.items():
            self.vocab[i] = self.vocab[a] + self.vocab[b]
        self.cache = {}

    def encode_chunk(self, chunk):
        if chunk not in self.cache:
            raise NotImplementedError("Write your code here")
        return self.cache[chunk]

    def encode(self, text):
        return [i for chunk in PATTERN.findall(text) for i in self.encode_chunk(chunk)]

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")


t0 = time.time()
tok = Tokenizer(train_bpe(train_text, VOCAB_SIZE))
train_ids = torch.tensor(tok.encode(train_text), dtype=torch.long)
val_ids = torch.tensor(tok.encode(val_text), dtype=torch.long)
chars_per_token = len(val_text) / len(val_ids)
print(f"tokenizer trained in {time.time() - t0:.0f} s; {chars_per_token:.2f} characters per token on validation text")
print([tok.vocab[i] for i in range(256, 266)])

## 2. The model

The same pre-norm GPT as in L13 (paste your own if you prefer), with the output layer tied to the
token embedding.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, n_head):
        super().__init__()
        self.n_head = n_head
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def attn(self, x):
        B, T, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=2)
        q, k, v = (t.view(B, T, self.n_head, d // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, d))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class GPT(nn.Module):
    def __init__(self, vocab, ctx, d, n_layer, n_head):
        super().__init__()
        self.ctx = ctx
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(ctx, d)
        self.blocks = nn.ModuleList([Block(d, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        # Small embeddings (as in GPT-2): with tied weights, N(0, 1) entries would make the initial
        # logits huge.
        nn.init.normal_(self.tok.weight, std=0.02)
        nn.init.normal_(self.pos.weight, std=0.02)

    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks:
            x = b(x)
        return self.ln_f(x) @ self.tok.weight.T

## 3. Training with evaluation

Write `estimate_loss`, the mean cross-entropy (nats per token) over `batches` random validation
windows drawn with a fixed seed, in evaluation mode and without gradients.

In [ ]:
CFG = dict(ctx=128, d=192, n_layer=4, n_head=6, batch=32, steps=1500, lr=2e-3, warmup=100)
if SMALL:
    CFG.update(ctx=32, d=32, n_layer=1, n_head=2, batch=8, steps=120, lr=3e-3, warmup=10)


def get_batch(ids, gen):
    ix = torch.randint(len(ids) - CFG["ctx"] - 1, (CFG["batch"],), generator=gen)
    x = torch.stack([ids[i:i + CFG["ctx"]] for i in ix])
    y = torch.stack([ids[i + 1:i + CFG["ctx"] + 1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)


@torch.no_grad()
def estimate_loss(model, ids, batches=20, seed=1):
    raise NotImplementedError("Write your code here")


model = GPT(len(tok.vocab), CFG["ctx"], CFG["d"], CFG["n_layer"], CFG["n_head"]).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.1)
gen = torch.Generator().manual_seed(0)
history = []
t0 = time.time()
for step in range(1, CFG["steps"] + 1):
    lr = CFG["lr"] * min(1.0, step / CFG["warmup"]) * 0.5 * (1.0 + math.cos(math.pi * step / CFG["steps"]))
    for group in opt.param_groups:
        group["lr"] = lr
    x, y = get_batch(train_ids, gen)
    loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step == 1 or step % max(1, CFG["steps"] // 10) == 0:
        history.append((step, loss.item(), estimate_loss(model, val_ids, batches=5)))
        print(f"step {step}: train {history[-1][1]:.3f}, val {history[-1][2]:.3f} nats/token ({time.time() - t0:.0f} s)")

val_nats_per_token = estimate_loss(model, val_ids)
val_bits_per_char = val_nats_per_token / chars_per_token / math.log(2)
print(f"validation: {val_nats_per_token:.3f} nats per token = {val_bits_per_char:.3f} bits per character")

## 4. Sampling

Write `filter_logits(logits, top_k, top_p)` for a 1-D tensor of logits: keep the `top_k` largest
(if given), then the smallest set of tokens, in order of probability, whose total probability reaches
`top_p` (if given), setting everything else to `-inf`. `generate` applies the temperature first.

In [ ]:
def filter_logits(logits, top_k=None, top_p=None):
    raise NotImplementedError("Write your code here")


@torch.no_grad()
def generate(prompt, n_tokens, temperature=1.0, top_k=None, top_p=None, seed=0):
    model.eval()
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    idx = torch.tensor([tok.encode(prompt)], device=DEVICE)
    for _ in range(n_tokens):
        logits = model(idx[:, -CFG["ctx"]:])[0, -1] / temperature
        probs = torch.softmax(filter_logits(logits, top_k, top_p), dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1, generator=g)[None]], dim=1)
    model.train()
    return tok.decode(idx[0].tolist())


samples = {T: generate("ROMEO:\n", 120, temperature=T, top_p=0.9) for T in (0.5, 1.0)}
for T, s in samples.items():
    print(f"--- temperature {T}, top-p 0.9 ---\n{s}\n")

## 5. Report

At least 80 words: your tokenizer's vocabulary size and characters per token, the model and training
settings, the validation loss in nats per token and in bits per character, and what changes in the
samples between the two temperatures.

In [ ]:
REPORT = """Write your report here."""

In [ ]:
# Forge autograder: run this cell last. It runs the capstone tests and prints your completion code.
import hashlib as _hashlib, sys as _sys, types as _types, unittest as _unittest
_FORGE_ID = "L14"
_FORGE_TESTS = '''import unittest
import torch
import sub


def test_tokenizer_roundtrip():
    for s in ["ROMEO:\\nBut, soft!", "  leading and trailing  \\n", "naïve café 東京", sub.val_text[:500]]:
        assert sub.tok.decode(sub.tok.encode(s)) == s


def test_tokenizer_compresses():
    assert len(sub.tok.vocab) == sub.VOCAB_SIZE
    assert sub.chars_per_token > (1.3 if sub.SMALL else 1.7), sub.chars_per_token


def test_filters():
    logits = torch.tensor([3.0, 2.0, 1.0, 0.0, -1.0], device=sub.DEVICE)
    k = sub.filter_logits(logits, top_k=2)
    assert torch.isfinite(k).sum() == 2 and torch.isfinite(k[:2]).all()
    p = sub.filter_logits(logits, top_p=0.9)
    probs = torch.softmax(logits, 0)
    assert torch.isfinite(p).sum() == 3 and probs[:2].sum() < 0.9 <= probs[:3].sum()
    assert torch.equal(sub.filter_logits(logits), logits)


def test_training_reduces_validation_loss():
    first, last = sub.history[0][2], sub.history[-1][2]
    assert last < first - 0.5, sub.history


def test_bits_per_character_target():
    if sub.SMALL:
        raise unittest.SkipTest("the target applies to the full training run")
    assert sub.val_bits_per_char < 2.3, sub.val_bits_per_char


def test_report():
    text = sub.REPORT.lower()
    assert len(text.split()) >= 80
    assert "bits per character" in text and "temperature" in text
'''
sub = _types.SimpleNamespace(**{_k: _v for _k, _v in globals().items() if not _k.startswith('_')})
_sys.modules['sub'] = sub
_ns = {'__name__': 'forge_tests'}
exec(compile(_FORGE_TESTS, 'tests.py', 'exec'), _ns)
_passed = []
for _name, _fn in list(_ns.items()):
    if _name.startswith('test_') and callable(_fn):
        try:
            _fn()
            _passed.append(_name)
            print('PASS', _name)
        except _unittest.SkipTest as _e:
            print('SKIP', _name, '-', _e)
        except Exception as _e:
            print('FAIL', _name, '-', type(_e).__name__, _e)
_digest = _hashlib.sha256(f"{_FORGE_ID}:{','.join(sorted(_passed))}:forge".encode()).hexdigest()[:8]
print(f"Completion code: FORGE-{_FORGE_ID}-{_digest}")